# Motor-Pore Offset Analysis: Characterizing Amino Acid Signatures

**Issue**: [#81 - Implement motor-pore offset correction for amino acid signatures](https://github.com/rnabioco/leech/issues/81)

## Background

Nanopore sequencers have two detection sites:
1. **Motor (helicase)**: Controls translocation speed → affects **dwell time**
2. **Pore (reader head)**: Measures ionic current → affects **electrical signal**

These sites are separated by **12-13 nucleotides** physically. When an aminoacylated tRNA passes through:
- Amino acid in **motor** → increases dwell time for nucleotides currently in **pore** (~12-13 nt away)
- Amino acid in **pore** → distorts electrical current

**Current Problem**: The codebase incorrectly assumes dwell and current signatures are aligned to the same genomic position. This explains why **dwell features don't improve model accuracy** - they're extracted from the wrong position.

## Analysis Goals

1. **Extract wide-window features** (±30 nt from motif) for charged vs uncharged tRNAs
2. **Visualize position-wise signatures** to identify where dwell and current changes occur
3. **Measure offset empirically** - determine direction and magnitude
4. **Propose implementation strategy** for offset-corrected feature extraction

## Molecule Structure

```
5' ─── [tRNA ~73 nt] ─── CCA-[aa]-GGC ─── [adaptor ~37 nt, AAAA] ─── 3'
                             ↑
                       Motif: CCATGGC
                       (T = amino acid placeholder in reference)

Sequencing direction: 3' → 5' (adaptor enters pore first)
Reporting: 5' → 3' (standard convention)
```

In [ ]:
# Imports
import warnings
from pathlib import Path

import numpy as np
import plotnine as p9
import polars as pl
from scipy import signal as sp_signal
from scipy import stats

# leech imports
from leech.features import (
    compute_dwell_times,
    compute_signal_features,
    extract_move_table,
    normalize_signal,
)
from leech.io.bam_reader import BAMReader
from leech.io.motif_search import get_motif_searcher
from leech.io.pod5_reader import POD5Reader
from leech.io.reference import get_reference_sequences

warnings.filterwarnings("ignore")

# Plotting aesthetics
p9.theme_set(p9.theme_minimal() + p9.theme(figure_size=(14, 6)))

print("✓ Imports successful")

## Configuration

Set paths to your POD5 and BAM files, and define extraction parameters.

In [ ]:
# ============================================================================
# USER CONFIGURATION - Update for your analysis
# ============================================================================

# All available charged amino acid samples
CHARGED_SAMPLES = [
    "ala_synthetic",
    "arg_synthetic",
    "asn_synthetic",
    "asp_synthetic",
    "cys_synthetic_rep1",
    "cys_synthetic_rep2",
    "gln_synthetic",
    "glu_synthetic",
    "gly_synthetic",
    "his_synthetic",
    "ile_synthetic",
    "leu_synthetic",
    "lys_synthetic",
    "met_synthetic",
    "phe_synthetic",
    "pro_synthetic",
    "ser_synthetic",
    "thr_synthetic",
    "trp_synthetic",
    "tyr_synthetic",
    "val_synthetic"
]

UNCHARGED_SAMPLE = "uncharged_synthetic"

# Pipeline paths (from pipeline/config/samples-alpine.yaml and config.yaml)
PROJECT_NAME = "synthetic-trna"  # From samples-alpine.yaml
BASE_DIR = Path("/scratch/alpine/jhesselberth@xsede.org/leech")

# Reference FASTA (for motif position identification)
REFERENCE_FASTA = Path("../pipeline/resources/references/synthetic-trna.fa")

# Motif search parameters (from pipeline config)
MOTIF = "CCATGGC"  # CCA-[aa as T]-GGC (T is placeholder for amino acid)
MOTIF_OFFSET = 3  # Focus on position 3 (the 'T' = amino acid position)
MOTIF_REFERENCE = "fasta"  # Search in reference sequence (avoids basecalling bias)

# Wide-window extraction parameters
KMER_CONTEXT = 30  # Extract ±30 bases from motif (total 61 bases)
SIGNAL_CONTEXT = (600, 600)  # Signal context (samples left, right)

# Quality filtering parameters
MIN_MAPQ = 30  # Minimum mapping quality (increase to reduce noise; default was 10)
# Higher MAPQ = higher quality alignments = less noisy signals
# MAPQ 30 = 99.9% probability of correct mapping
# MAPQ 20 = 99% probability
# MAPQ 10 = 90% probability

# Sampling parameters
MAX_READS_PER_CLASS = 500  # Number of reads to sample from each class
RANDOM_SEED = 42  # For reproducibility

# Output directory
OUTPUT_DIR = Path("../output/offset_analysis") / "all_amino_acids"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("MOTOR-PORE OFFSET ANALYSIS CONFIGURATION")
print("=" * 80)
print(f"\nAnalyzing {len(CHARGED_SAMPLES)} amino acid samples:")
for i, aa in enumerate(CHARGED_SAMPLES, 1):
    print(f"  {i:2d}. {aa}")
print(f"\nUncharged control: {UNCHARGED_SAMPLE}")

print("\nData sources (pipeline-generated files):")
print(f"  Base directory: {BASE_DIR / PROJECT_NAME}")
print("  POD5 pattern:   pod5/{sample}/{sample}.pod5")
print("  BAM pattern:    bam/rebasecall/{sample}/{sample}.aligned.bam")
print(f"  Reference:      {REFERENCE_FASTA}")

print("\nQuality filtering:")
print(f"  Min MAPQ:        {MIN_MAPQ} (higher = better quality, less noise)")

print("\nAnalysis parameters:")
print(f"  Motif:           {MOTIF} (offset={MOTIF_OFFSET}, reference-based)")
print(f"  K-mer context:   ±{KMER_CONTEXT} bases (total {2*KMER_CONTEXT+1} bases)")
print(f"  Signal context:  {SIGNAL_CONTEXT[0]}/{SIGNAL_CONTEXT[1]} samples")
print(f"  Max reads:       {MAX_READS_PER_CLASS} per class")

print("\nOutput:")
print(f"  Directory:       {OUTPUT_DIR}")
print("=" * 80)

# Verify reference FASTA exists
print("\nVerifying reference FASTA...")
if REFERENCE_FASTA.exists():
    print(f"  ✓ Reference: {REFERENCE_FASTA}")
else:
    print(f"  ✗ Reference: {REFERENCE_FASTA} (NOT FOUND)")
    print("   ⚠️  WARNING: Reference FASTA is missing!")

print("\n✓ Configuration complete. Ready to proceed.")

# ============================================================================
# MAPQ DISTRIBUTION ANALYSIS
# ============================================================================
# Visualize alignment quality scores to help set MIN_MAPQ threshold

print("=" * 80)
print("ANALYZING MAPQ DISTRIBUTIONS")
print("=" * 80)

mapq_data = []
n_samples_to_check = min(5, len(CHARGED_SAMPLES))  # Check first 5 samples for speed

print(f"\nSampling MAPQ scores from {n_samples_to_check} amino acid samples...")
print("(This helps determine a good quality threshold)")

for aa_sample in CHARGED_SAMPLES[:n_samples_to_check]:
    bam_path = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / aa_sample / f"{aa_sample}.aligned.bam"
    
    if not bam_path.exists():
        print(f"  ⚠️  Skipping {aa_sample}: BAM not found")
        continue
    
    print(f"  Reading {aa_sample}...")
    
    # Read MAPQ scores from BAM (sample first 10000 reads for speed)
    bam_reader = BAMReader(bam_path, min_mapq=0)  # No filtering yet
    
    n_reads = 0
    max_reads = 10000
    
    with bam_reader:
        for aln in bam_reader.iter_alignments():
            if n_reads >= max_reads:
                break
            
            if not aln.is_unmapped and aln.mapping_quality is not None:
                mapq_data.append({
                    "sample": aa_sample.replace("_synthetic", "").upper(),
                    "mapq": aln.mapping_quality
                })
                n_reads += 1

# Also check uncharged
uncharged_bam = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / UNCHARGED_SAMPLE / f"{UNCHARGED_SAMPLE}.aligned.bam"
if uncharged_bam.exists():
    print(f"  Reading {UNCHARGED_SAMPLE}...")
    bam_reader = BAMReader(uncharged_bam, min_mapq=0)
    n_reads = 0
    with bam_reader:
        for aln in bam_reader.iter_alignments():
            if n_reads >= max_reads:
                break
            if not aln.is_unmapped and aln.mapping_quality is not None:
                mapq_data.append({
                    "sample": "UNCHARGED",
                    "mapq": aln.mapping_quality
                })
                n_reads += 1

mapq_df = pl.DataFrame(mapq_data).to_pandas()

print(f"\n✓ Collected {len(mapq_df)} MAPQ scores from {mapq_df['sample'].nunique()} samples")

# Compute summary statistics
print("\nMAPQ Summary Statistics:")
print(mapq_df.groupby("sample")["mapq"].describe())

# === VIOLIN PLOT ===
plot1 = (
    p9.ggplot(mapq_df, p9.aes(x="sample", y="mapq", fill="sample"))
    + p9.geom_violin(alpha=0.7)
    + p9.geom_boxplot(width=0.1, alpha=0.5)
    + p9.geom_hline(yintercept=10, linetype="dashed", color="orange", alpha=0.5)
    + p9.geom_hline(yintercept=20, linetype="dashed", color="yellow", alpha=0.5)
    + p9.geom_hline(yintercept=30, linetype="dashed", color="green", alpha=0.5)
    + p9.annotate("text", x=0.5, y=10, label="MAPQ=10 (90%)", ha="left", size=8, color="orange")
    + p9.annotate("text", x=0.5, y=20, label="MAPQ=20 (99%)", ha="left", size=8, color="gold")
    + p9.annotate("text", x=0.5, y=30, label="MAPQ=30 (99.9%)", ha="left", size=8, color="green")
    + p9.labs(
        title="MAPQ Distribution by Sample",
        x="Sample",
        y="Mapping Quality (MAPQ)",
        fill="Sample"
    )
    + p9.theme_minimal()
    + p9.theme(
        figure_size=(12, 6),
        plot_title=p9.element_text(size=13, weight="bold"),
        axis_text_x=p9.element_text(angle=45, ha="right", size=8),
        legend_position="none"
    )
)

plot1.show()

# === HISTOGRAM ===
plot2 = (
    p9.ggplot(mapq_df, p9.aes(x="mapq"))
    + p9.geom_histogram(binwidth=5, fill="#3498DB", alpha=0.7, color="white")
    + p9.geom_vline(xintercept=10, linetype="dashed", color="orange", size=1)
    + p9.geom_vline(xintercept=20, linetype="dashed", color="gold", size=1)
    + p9.geom_vline(xintercept=30, linetype="dashed", color="green", size=1)
    + p9.facet_wrap("~sample", ncol=3)
    + p9.labs(
        title="MAPQ Distribution (by sample)",
        x="Mapping Quality (MAPQ)",
        y="Count"
    )
    + p9.theme_minimal()
    + p9.theme(
        figure_size=(14, 8),
        plot_title=p9.element_text(size=13, weight="bold"),
        strip_text=p9.element_text(size=9, weight="bold")
    )
)

plot2.show()

# Recommendations
print("\n" + "=" * 80)
print("MAPQ THRESHOLD RECOMMENDATIONS")
print("=" * 80)

pct_above_10 = (mapq_df["mapq"] >= 10).sum() / len(mapq_df) * 100
pct_above_20 = (mapq_df["mapq"] >= 20).sum() / len(mapq_df) * 100
pct_above_30 = (mapq_df["mapq"] >= 30).sum() / len(mapq_df) * 100

print(f"\nPercentage of reads above threshold:")
print(f"  MAPQ ≥ 10: {pct_above_10:.1f}% (90% mapping accuracy)")
print(f"  MAPQ ≥ 20: {pct_above_20:.1f}% (99% mapping accuracy)")
print(f"  MAPQ ≥ 30: {pct_above_30:.1f}% (99.9% mapping accuracy)")

print(f"\nCurrent MIN_MAPQ setting: {MIN_MAPQ}")
print(f"→ This will retain ~{(mapq_df['mapq'] >= MIN_MAPQ).sum() / len(mapq_df) * 100:.1f}% of reads")

print("\nRecommendations:")
print("  • MIN_MAPQ = 10: Keep most reads, moderate quality")
print("  • MIN_MAPQ = 20: Good balance of quality vs quantity")
print("  • MIN_MAPQ = 30: Highest quality, fewer reads (may reduce noise)")
print("\n  Adjust MIN_MAPQ in the configuration cell above if needed.")
print("=" * 80)

In [ ]:
def extract_wide_window_features(pod5_path, bam_path, label_name, max_reads=500, reference_fasta=None):
    """
    Extract wide-window features for offset analysis.

    Returns:
        List of dicts with features at each position relative to motif.
    """
    print(f"\nExtracting features from {label_name}...")
    print(f"  POD5: {pod5_path}")
    print(f"  BAM:  {bam_path}")

    # Load reference sequences and create motif searcher
    reference_sequences = get_reference_sequences(bam_path, reference_fasta)
    print(f"  Loaded {len(reference_sequences)} reference sequences")

    motif_searcher = get_motif_searcher(
        mode=MOTIF_REFERENCE,
        reference_sequences=reference_sequences,
        skip_indels=True
    )

    chunks = []
    bam_reader = BAMReader(bam_path, min_mapq=MIN_MAPQ)
    pod5_reader = POD5Reader(pod5_path)

    n_processed = 0
    n_with_motif = 0
    n_errors = 0

    with bam_reader, pod5_reader:
        for aln in bam_reader.iter_alignments():
            if len(chunks) >= max_reads:
                break

            n_processed += 1

            if aln.is_unmapped or not aln.has_tag("mv") or aln.query_name is None:
                continue

            if aln.query_sequence is None:
                continue

            try:
                # Find motif positions using motif searcher
                motif_positions = motif_searcher.find_motif_positions(
                    read_id=aln.query_name,
                    sequence=aln.query_sequence,
                    alignment=aln,
                    motif=MOTIF
                )

                if not motif_positions:
                    continue

                n_with_motif += 1

                # Get signal and features
                signal, _ = pod5_reader.get_signal(aln.query_name)
                signal_norm = normalize_signal(signal, method="median_mad")

                move_table = extract_move_table(aln)
                seq_to_sig_map = move_table.to_seq_to_sig_map()

                # Compute dwell times and signal features
                dwells = compute_dwell_times(move_table)
                signal_features = compute_signal_features(signal_norm, seq_to_sig_map)

                # Extract wide window around each motif occurrence
                for motif_pos in motif_positions:
                    focus_pos = motif_pos + MOTIF_OFFSET

                    # Check boundaries
                    start_pos = focus_pos - KMER_CONTEXT
                    end_pos = focus_pos + KMER_CONTEXT + 1

                    if start_pos < 0 or end_pos > len(dwells):
                        continue

                    # Extract features across window
                    chunk = {
                        "read_id": aln.query_name,
                        "label": label_name,
                        "motif_pos": motif_pos,
                        "focus_pos": focus_pos,
                        "sequence": aln.query_sequence[start_pos:end_pos],
                        "dwells": dwells[start_pos:end_pos],
                        "signal_mean": signal_features["level_mean"][start_pos:end_pos],
                        "signal_std": signal_features["level_std"][start_pos:end_pos],
                        "signal_median": signal_features["level_median"][start_pos:end_pos],
                    }
                    chunks.append(chunk)

                    if len(chunks) >= max_reads:
                        break

            except Exception as e:
                n_errors += 1
                if n_errors <= 5:  # Only print first 5 errors
                    print(f"  Warning: Failed to process {aln.query_name}: {e}")
                continue

    if n_errors > 5:
        print(f"  ... and {n_errors - 5} more errors (suppressed)")

    print(f"  Processed {n_processed} alignments")
    print(f"  Found motif in {n_with_motif} reads")
    print(f"  Extracted {len(chunks)} wide-window chunks")

    return chunks


# Extract features for all samples
print("=" * 80)
print("EXTRACTING WIDE-WINDOW FEATURES")
print("=" * 80)

all_chunks = {}

# Extract uncharged once
uncharged_pod5 = BASE_DIR / PROJECT_NAME / "pod5" / UNCHARGED_SAMPLE / f"{UNCHARGED_SAMPLE}.pod5"
uncharged_bam = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / UNCHARGED_SAMPLE / f"{UNCHARGED_SAMPLE}.aligned.bam"

print("\nExtracting uncharged control...")
uncharged_chunks = extract_wide_window_features(
    uncharged_pod5, uncharged_bam, "uncharged",
    max_reads=MAX_READS_PER_CLASS,
    reference_fasta=REFERENCE_FASTA
)
all_chunks["uncharged"] = uncharged_chunks

# Extract each charged amino acid
for aa_sample in CHARGED_SAMPLES:
    charged_pod5 = BASE_DIR / PROJECT_NAME / "pod5" / aa_sample / f"{aa_sample}.pod5"
    charged_bam = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / aa_sample / f"{aa_sample}.aligned.bam"

    charged_chunks = extract_wide_window_features(
        charged_pod5, charged_bam, aa_sample,
        max_reads=MAX_READS_PER_CLASS,
        reference_fasta=REFERENCE_FASTA
    )
    all_chunks[aa_sample] = charged_chunks

total_chunks = sum(len(chunks) for chunks in all_chunks.values())
print(f"\n✓ Total chunks extracted: {total_chunks}")
print(f"  Samples: {list(all_chunks.keys())}")

## Step 2: Compute Position-Wise Statistics

Aggregate features across all reads to compute mean and std at each position relative to the motif.

In [ ]:
def aggregate_position_wise_features(chunks, label):
    """
    Aggregate features across all chunks to get position-wise statistics.

    Returns:
        DataFrame with columns: position, feature, mean, std, sem, label
    """
    print(f"\nAggregating features for {label} ({len(chunks)} chunks)...")

    # Stack all features into arrays
    window_size = 2 * KMER_CONTEXT + 1

    dwells_matrix = np.array([c["dwells"] for c in chunks if len(c["dwells"]) == window_size])
    signal_mean_matrix = np.array([c["signal_mean"] for c in chunks if len(c["signal_mean"]) == window_size])
    signal_std_matrix = np.array([c["signal_std"] for c in chunks if len(c["signal_std"]) == window_size])

    print(f"  Valid chunks after filtering: {len(dwells_matrix)}")

    # Compute position-wise statistics
    positions = np.arange(-KMER_CONTEXT, KMER_CONTEXT + 1)  # Relative to motif focus

    data = []

    for pos_idx, pos in enumerate(positions):
        # Dwell
        data.append({
            "position": pos,
            "feature": "dwell_time",
            "mean": np.mean(dwells_matrix[:, pos_idx]),
            "std": np.std(dwells_matrix[:, pos_idx]),
            "sem": stats.sem(dwells_matrix[:, pos_idx]),
            "label": label,
        })

        # Signal mean
        data.append({
            "position": pos,
            "feature": "signal_mean",
            "mean": np.mean(signal_mean_matrix[:, pos_idx]),
            "std": np.std(signal_mean_matrix[:, pos_idx]),
            "sem": stats.sem(signal_mean_matrix[:, pos_idx]),
            "label": label,
        })

        # Signal std (variability)
        data.append({
            "position": pos,
            "feature": "signal_variability",
            "mean": np.mean(signal_std_matrix[:, pos_idx]),
            "std": np.std(signal_std_matrix[:, pos_idx]),
            "sem": stats.sem(signal_std_matrix[:, pos_idx]),
            "label": label,
        })

    return pl.DataFrame(data)


# Aggregate for all samples
print("=" * 80)
print("COMPUTING POSITION-WISE STATISTICS")
print("=" * 80)

all_stats_list = []

for sample_name, chunks in all_chunks.items():
    sample_stats = aggregate_position_wise_features(chunks, sample_name)
    all_stats_list.append(sample_stats)

# Combine all
all_stats = pl.concat(all_stats_list)

print(f"\n✓ Aggregated statistics: {len(all_stats)} rows")
print(f"  Samples: {all_stats['label'].unique().to_list()}")
print(f"  Features: {all_stats['feature'].unique().to_list()}")
print(f"  Position range: [{all_stats['position'].min()}, {all_stats['position'].max()}]")

# Preview
print("\nPreview (first 10 rows):")
print(all_stats.head(10))

## Step 3: Visualization - Position-Wise Feature Profiles

Plot dwell time and signal mean across position for charged vs uncharged.
This will reveal where the signatures appear relative to the motif (position 0).

In [ ]:
# Convert to pandas for plotting
stats_pd = all_stats.to_pandas()

# Create plots for each feature
features_to_plot = ["dwell_time", "signal_mean", "signal_variability"]
titles = {
    "dwell_time": "Dwell Time Across Position (Motor Signature)",
    "signal_mean": "Signal Mean Across Position (Pore Signature)",
    "signal_variability": "Signal Variability Across Position"
}

# === HEATMAPS: Show all amino acids at once ===
print("=" * 80)
print("HEATMAP VISUALIZATIONS")
print("=" * 80)

for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature].copy()

    # Clean up sample names for better display
    feature_data["sample"] = feature_data["label"].apply(
        lambda x: "UNCHARGED" if x == "uncharged" else x.replace("_synthetic", "").upper()
    )

    # Create heatmap
    plot = (
        p9.ggplot(feature_data, p9.aes(x="position", y="sample", fill="mean"))
        + p9.geom_tile()
        + p9.scale_fill_gradient2(
            low="#3498DB", mid="#FFFFFF", high="#E74C3C",
            midpoint=feature_data["mean"].median()
        )
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.8, size=0.8)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="gray", alpha=0.5)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="gray", alpha=0.5)
        + p9.labs(
            title=f"{titles[feature]} - Heatmap",
            x="Position Relative to Motif (nt)",
            y="Sample",
            fill=feature.replace("_", " ").title()
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(16, 10),
            plot_title=p9.element_text(size=13, weight="bold"),
            axis_text_y=p9.element_text(size=8)
        )
    )

    plot.show()

# === LINE PLOTS: Uncharged vs all charged (averaged) ===
print("\n" + "=" * 80)
print("LINE PLOT: AVERAGE CHARGED VS UNCHARGED")
print("=" * 80)

for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature].copy()

    # Compute average across all charged samples
    charged_avg = feature_data[feature_data["label"] != "uncharged"].groupby("position").agg({
        "mean": "mean",
        "sem": lambda x: np.sqrt(np.sum(x**2)) / len(x)  # Combined SEM
    }).reset_index()
    charged_avg["label"] = "Charged (avg)"

    # Get uncharged data
    uncharged_data = feature_data[feature_data["label"] == "uncharged"][["position", "mean", "sem"]].copy()
    uncharged_data["label"] = "Uncharged"

    # Combine
    combined = pl.concat([
        pl.DataFrame(charged_avg),
        pl.DataFrame(uncharged_data)
    ]).to_pandas()

    plot = (
        p9.ggplot(combined, p9.aes(x="position", y="mean", color="label", fill="label"))
        + p9.geom_line(size=1.5, alpha=0.9)
        + p9.geom_ribbon(p9.aes(ymin="mean - sem", ymax="mean + sem"), alpha=0.2, color=None)
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.5, size=1)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="gray", alpha=0.3)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="gray", alpha=0.3)
        + p9.scale_color_manual(values={"Charged (avg)": "#E74C3C", "Uncharged": "#95A5A6"})
        + p9.scale_fill_manual(values={"Charged (avg)": "#E74C3C", "Uncharged": "#95A5A6"})
        + p9.labs(
            title=titles[feature],
            x="Position Relative to Motif (nt)",
            y=feature.replace("_", " ").title(),
            color="Sample",
            fill="Sample"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 6),
            plot_title=p9.element_text(size=12, weight="bold"),
            legend_position="right"
        )
    )

    plot.show()

## Step 4: Difference Plots - Charged - Uncharged

Plot the difference (charged - uncharged) to highlight where signatures are strongest.

In [ ]:
# Compute differences (each charged - uncharged)
diff_data = []

# Get uncharged data for each feature
for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature]
    uncharged_data = feature_data[feature_data["label"] == "uncharged"].sort_values("position")

    # Compute difference for each charged amino acid
    for aa_sample in CHARGED_SAMPLES:
        aa_data = feature_data[feature_data["label"] == aa_sample].sort_values("position")

        if len(aa_data) == 0:
            continue

        positions = aa_data["position"].values
        diff = aa_data["mean"].values - uncharged_data["mean"].values

        for pos, d in zip(positions, diff, strict=False):
            diff_data.append({
                "position": pos,
                "feature": feature,
                "amino_acid": aa_sample,
                "difference": d
            })

diff_df = pl.DataFrame(diff_data).to_pandas()

# === DIFFERENCE HEATMAPS ===
print("=" * 80)
print("DIFFERENCE HEATMAPS (Charged - Uncharged)")
print("=" * 80)

for feature in features_to_plot:
    feature_diff = diff_df[diff_df["feature"] == feature].copy()

    # Clean up amino acid names
    feature_diff["aa"] = feature_diff["amino_acid"].apply(
        lambda x: x.replace("_synthetic", "").upper()
    )

    # Create heatmap
    plot = (
        p9.ggplot(feature_diff, p9.aes(x="position", y="aa", fill="difference"))
        + p9.geom_tile()
        + p9.scale_fill_gradient2(
            low="#3498DB", mid="#FFFFFF", high="#E74C3C",
            midpoint=0,
            limits=(-feature_diff["difference"].abs().max(), feature_diff["difference"].abs().max())
        )
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.8, size=0.8)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="black", alpha=0.5)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="black", alpha=0.5)
        + p9.labs(
            title=f"{titles[feature]} - Difference from Uncharged",
            x="Position Relative to Motif (nt)",
            y="Amino Acid",
            fill=f"Δ {feature.replace('_', ' ').title()}"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(16, 10),
            plot_title=p9.element_text(size=13, weight="bold"),
            axis_text_y=p9.element_text(size=8)
        )
    )

    plot.show()

# === AVERAGE DIFFERENCE LINE PLOT ===
print("\n" + "=" * 80)
print("AVERAGE DIFFERENCE (Mean across all amino acids)")
print("=" * 80)

for feature in features_to_plot:
    feature_diff = diff_df[diff_df["feature"] == feature].copy()

    # Compute average difference across all amino acids
    avg_diff = feature_diff.groupby("position").agg({
        "difference": ["mean", "std"]
    }).reset_index()
    avg_diff.columns = ["position", "mean", "std"]
    avg_diff["sem"] = avg_diff["std"] / np.sqrt(len(CHARGED_SAMPLES))

    plot = (
        p9.ggplot(avg_diff, p9.aes(x="position", y="mean"))
        + p9.geom_ribbon(p9.aes(ymin="mean - sem", ymax="mean + sem"), fill="#E74C3C", alpha=0.2)
        + p9.geom_line(color="#E74C3C", size=1.5, alpha=0.9)
        + p9.geom_hline(yintercept=0, linetype="solid", color="black", alpha=0.5, size=0.8)
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.5, size=1)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="gray", alpha=0.3)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="gray", alpha=0.3)
        + p9.labs(
            title=f"{titles[feature]} - Average Difference",
            x="Position Relative to Motif (nt)",
            y=f"Δ {feature.replace('_', ' ').title()}\n(Charged - Uncharged, mean ± SEM)"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 6),
            plot_title=p9.element_text(size=12, weight="bold")
        )
    )

    plot.show()

## Step 5: Peak Detection - Measure Offset

Automatically detect where dwell and current signatures peak, then calculate the offset.

In [ ]:
def find_signature_peak(diff_df, feature, amino_acid, window=(-20, 20)):
    """
    Find the position of maximum absolute difference for a feature.

    Args:
        diff_df: DataFrame with position and difference columns
        feature: Feature name to analyze
        amino_acid: Amino acid sample name
        window: (min_pos, max_pos) to search within

    Returns:
        dict with peak_position, peak_value, peak_type
    """
    feature_diff = diff_df[
        (diff_df["feature"] == feature) &
        (diff_df["amino_acid"] == amino_acid) &
        (diff_df["position"] >= window[0]) &
        (diff_df["position"] <= window[1])
    ].copy()

    if len(feature_diff) == 0:
        return None

    # Find position of maximum absolute difference
    feature_diff["abs_diff"] = feature_diff["difference"].abs()
    peak_idx = feature_diff["abs_diff"].idxmax()

    peak_row = feature_diff.loc[peak_idx]

    return {
        "amino_acid": amino_acid,
        "feature": feature,
        "peak_position": int(peak_row["position"]),
        "peak_value": float(peak_row["difference"]),
        "peak_type": "increase" if peak_row["difference"] > 0 else "decrease",
    }


# Detect peaks for each amino acid and feature
print("=" * 80)
print("PEAK DETECTION - SIGNATURE LOCALIZATION")
print("=" * 80)

all_peaks = []

for aa_sample in CHARGED_SAMPLES:
    print(f"\n{aa_sample.upper().replace('_SYNTHETIC', '')}:")
    aa_peaks = {}

    for feature in features_to_plot:
        peak_info = find_signature_peak(diff_df, feature, aa_sample, window=(-25, 25))
        if peak_info:
            all_peaks.append(peak_info)
            aa_peaks[feature] = peak_info

            print(f"  {feature}:")
            print(f"    Peak at position: {peak_info['peak_position']:+d} nt")
            print(f"    Peak value:       {peak_info['peak_value']:+.4f}")

    # Calculate offset for this amino acid
    if "dwell_time" in aa_peaks and "signal_mean" in aa_peaks:
        dwell_peak = aa_peaks["dwell_time"]["peak_position"]
        signal_peak = aa_peaks["signal_mean"]["peak_position"]
        measured_offset = abs(dwell_peak - signal_peak)

        print(f"  Measured offset: {measured_offset} nt (dwell={dwell_peak:+d}, current={signal_peak:+d})")

peaks_df = pl.DataFrame(all_peaks)

# Summary table of offsets
print("\n" + "=" * 80)
print("OFFSET SUMMARY ACROSS AMINO ACIDS")
print("=" * 80)

offset_summary = []
for aa_sample in CHARGED_SAMPLES:
    aa_peaks_df = peaks_df.filter(pl.col("amino_acid") == aa_sample)

    dwell_peaks = aa_peaks_df.filter(pl.col("feature") == "dwell_time")
    signal_peaks = aa_peaks_df.filter(pl.col("feature") == "signal_mean")

    if len(dwell_peaks) > 0 and len(signal_peaks) > 0:
        dwell_pos = dwell_peaks["peak_position"][0]
        signal_pos = signal_peaks["peak_position"][0]
        offset = abs(dwell_pos - signal_pos)

        offset_summary.append({
            "amino_acid": aa_sample.replace("_synthetic", "").upper(),
            "dwell_peak": dwell_pos,
            "current_peak": signal_pos,
            "offset_nt": offset
        })

offset_summary_df = pl.DataFrame(offset_summary)
print(offset_summary_df)

# Calculate mean offset
mean_offset = offset_summary_df["offset_nt"].mean()
std_offset = offset_summary_df["offset_nt"].std()

print(f"\n➤ MEAN OFFSET: {mean_offset:.1f} ± {std_offset:.1f} nt")
print("  Expected: ~12-13 nt")

if mean_offset >= 10 and mean_offset <= 15:
    print("\n✓ Measured offset is consistent with expected ~12-13 nt!")
else:
    print("\n⚠️  Measured offset differs from expected ~12-13 nt")

# Save measurements
offset_report = {
    "mean_offset_nt": float(mean_offset),
    "std_offset_nt": float(std_offset),
    "per_amino_acid": offset_summary_df.to_dicts(),
    "expected_offset_nt": "12-13",
}

import json

with open(OUTPUT_DIR / "offset_measurements.json", "w") as f:
    json.dump(offset_report, f, indent=2)

print(f"\n✓ Saved: {OUTPUT_DIR / 'offset_measurements.json'}")

## Step 6: Heatmap Visualization

Create a 2D heatmap showing all features across position for charged vs uncharged.

In [ ]:
# === OFFSET ANNOTATION HEATMAP ===
# Show where dwell and current peaks are for each amino acid

print("=" * 80)
print("PEAK POSITION HEATMAP")
print("=" * 80)

# Create a summary showing peak positions for each amino acid
peak_position_data = []

for aa_sample in CHARGED_SAMPLES:
    aa_peaks_df = peaks_df.filter(pl.col("amino_acid") == aa_sample)

    if len(aa_peaks_df) == 0:
        continue

    aa_name = aa_sample.replace("_synthetic", "").upper()

    for feature in ["dwell_time", "signal_mean"]:
        feature_peak = aa_peaks_df.filter(pl.col("feature") == feature)
        if len(feature_peak) > 0:
            peak_position_data.append({
                "amino_acid": aa_name,
                "feature": "Dwell" if feature == "dwell_time" else "Current",
                "peak_position": feature_peak["peak_position"][0]
            })

if len(peak_position_data) > 0:
    peak_pos_df = pl.DataFrame(peak_position_data).to_pandas()

    plot = (
        p9.ggplot(peak_pos_df, p9.aes(x="feature", y="amino_acid", fill="peak_position"))
        + p9.geom_tile(color="white", size=0.5)
        + p9.geom_text(p9.aes(label="peak_position"), size=8, color="black")
        + p9.scale_fill_gradient2(
            low="#3498DB", mid="#FFFFFF", high="#E74C3C",
            midpoint=0
        )
        + p9.labs(
            title="Peak Positions: Dwell vs Current Signatures",
            x="Signature Type",
            y="Amino Acid",
            fill="Position (nt)"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(6, 12),
            plot_title=p9.element_text(size=13, weight="bold"),
            axis_text_y=p9.element_text(size=8)
        )
    )

    plot.show()
else:
    print("No peak position data available yet (run peak detection cell first)")

## Step 7: Cross-Correlation Analysis

Compute cross-correlation between dwell and current difference signals to quantify the offset.

In [ ]:
# Compute cross-correlation for each amino acid
print("=" * 80)
print("CROSS-CORRELATION ANALYSIS")
print("=" * 80)

cross_corr_results = []

for aa_sample in CHARGED_SAMPLES:
    # Extract difference signals for this amino acid
    dwell_diff = diff_df[
        (diff_df["feature"] == "dwell_time") &
        (diff_df["amino_acid"] == aa_sample)
    ].sort_values("position")

    signal_diff = diff_df[
        (diff_df["feature"] == "signal_mean") &
        (diff_df["amino_acid"] == aa_sample)
    ].sort_values("position")

    if len(dwell_diff) == 0 or len(signal_diff) == 0:
        continue

    # Compute cross-correlation
    correlation = sp_signal.correlate(
        dwell_diff["difference"],
        signal_diff["difference"],
        mode="full"
    )
    lags = sp_signal.correlation_lags(
        len(dwell_diff["difference"]),
        len(signal_diff["difference"]),
        mode="full"
    )

    # Find peak correlation
    peak_idx = np.argmax(np.abs(correlation))
    peak_lag = lags[peak_idx]
    peak_corr = correlation[peak_idx]

    aa_name = aa_sample.replace("_synthetic", "").upper()
    print(f"\n{aa_name}:")
    print(f"  Peak correlation: {peak_corr:.4f}")
    print(f"  Peak lag:         {peak_lag} positions")

    cross_corr_results.append({
        "amino_acid": aa_name,
        "peak_lag": peak_lag,
        "peak_correlation": peak_corr
    })

# Summary
print("\n" + "=" * 80)
print("CROSS-CORRELATION SUMMARY")
print("=" * 80)
cross_corr_df = pl.DataFrame(cross_corr_results)
print(cross_corr_df)

mean_lag = cross_corr_df["peak_lag"].mean()
print(f"\n➤ MEAN LAG: {mean_lag:.1f} positions")
print(f"  Interpretation: Dwell signal is shifted by ~{mean_lag:.0f} positions relative to current signal")

## Summary and Recommendations

Based on the analysis above, we can now make data-driven decisions about implementing offset correction.

In [ ]:
print("=" * 80)
print("SUMMARY & RECOMMENDATIONS")
print("=" * 80)

print(f"\n1. MEASURED OFFSET ACROSS {len(CHARGED_SAMPLES)} AMINO ACIDS:")
print(f"   Mean offset: {mean_offset:.1f} ± {std_offset:.1f} nt")
print("   Expected:    ~12-13 nt")
print("\n   Per amino acid:")
for row in offset_summary_df.to_dicts():
    print(f"     {row['amino_acid']:3s}: {row['offset_nt']:2d} nt (dwell={row['dwell_peak']:+3d}, current={row['current_peak']:+3d})")

print("\n2. BIOLOGICAL INTERPRETATION:")
print("   The consistent offset across different amino acids suggests:")
print("   • Motor-pore offset is a structural property of the nanopore")
print("   • Not dependent on specific amino acid chemistry")
print("   • Should apply universally to all aa-tRNA modifications")

print("\n3. IMPLEMENTATION RECOMMENDATIONS:")
print("\n   A. IMMEDIATE ACTION:")
print("      Modify chunking/extractor.py to extract features from offset windows")
print(f"      - Use mean dwell offset: ~{mean_offset:.0f} nt")
print("      - Extract dwell and current features from different genomic positions")

print("\n   B. CONFIGURATION:")
print("      Add parameters to constants.py or config:")
print(f"      - MOTOR_PORE_OFFSET = {int(mean_offset)}")
print("      - Apply offset when extracting dwell-based features")

print("\n   C. MODEL ARCHITECTURE:")
if mean_offset > 10:
    print("      Option 1 (Recommended): Offset-aware feature extraction")
    print("        - Extract dwell features from base_idx + offset")
    print("        - Extract current features from base_idx")
    print("        - Keep current model architecture")
    print("")
    print("      Option 2: Wider context window")
    print(f"        - Use context ±{int(mean_offset) + 5} nt to capture both signatures")
    print("        - Let convolutions/attention learn the offset")

print("\n   D. VALIDATION:")
print("      1. Implement offset-corrected feature extraction")
print("      2. Prepare new training data with corrected offsets")
print("      3. Retrain ConvLSTMDwell model")
print("      4. Compare accuracy: baseline vs offset-aware")
print("      5. EXPECT: Significant improvement in dwell feature contribution")

print("\n4. NEXT STEPS:")
print("   [ ] Implement offset correction in chunking/extractor.py")
print("   [ ] Update prepare command to support --motor-pore-offset")
print("   [ ] Regenerate training data with offset correction")
print("   [ ] Retrain all models with corrected features")
print("   [ ] Benchmark old vs new models")
print("   [ ] Update documentation in issue #81")

print("\n" + "=" * 80)